# Diseño de aplicaciones de IA con LLM
## Módulo: Seguridad y cumplimiento

- **Institución:** BSG Institute
- **Curso:** Diseño de aplicaciones de IA con LLM
- **Módulo:** Seguridad y cumplimiento de datos
- **Docente:** Jorge I. Blanco

En este notebook vamos a usar **Named Entity Recognition (NER)** en español para detectar datos personales y sensibles en texto, y conectar estos resultados con marcos regulatorios como:

- **GDPR** (*General Data Protection Regulation*), aplicable a datos personales de personas en la Unión Europea.
- **Ley 1581 de 2012** en Colombia, sobre protección de datos personales.
- **LFPDPPP** en México (*Ley Federal de Protección de Datos Personales en Posesión de los Particulares*).
- **Ley 29733** en Perú, Ley de Protección de Datos Personales.

La idea es avanzar desde ejemplos simples (detectar nombres, organizaciones y lugares) hasta textos de dominios sensibles como salud y legal, manteniendo el enfoque en **seguridad y cumplimiento**.

## Índice

- [1. Marco de privacidad: GDPR y leyes locales](#sec-marco-privacidad)
- [2. Recordatorio técnico: Token classification y NER](#sec-recordatorio-tecnico)
- [3. Selección de modelo NER en español](#sec-modelo-ner)
- [4. Ejercicio 1 – NER básico en español](#sec-ejercicio-1)
- [5. Ejercicio 2 – Anonimización básica con NER](#sec-ejercicio-2)
- [6. Ejercicio 3 – Frases con mayor complejidad](#sec-ejercicio-3)
- [7. Ejercicio 4 – Rellenar máscaras con `fill-mask`](#sec-ejercicio-4)
- [8. Ejercicio 5 – NER aplicado a salud / medicina](#sec-ejercicio-5)
- [9. Ejercicio 6 – NER aplicado a documentos legales](#sec-ejercicio-6)
- [10. Ejercicio 7 – Selección autónoma de modelo y mini-experimento](#sec-ejercicio-7)
- [11. Ejercicio 8 – Caso de estudio de cumplimiento](#sec-ejercicio-8)
- [12. Recursos y datasets recomendados](#sec-recursos)
- [13. Cierre y reflexión final](#sec-cierre)


<a id="sec-marco-privacidad"></a>
## 2. Marco de privacidad: GDPR y leyes locales

Antes de escribir una sola línea de código, es importante tener claro **por qué** estamos haciendo todo esto.

### 2.1. GDPR (General Data Protection Regulation)

El **GDPR** es el reglamento europeo de protección de datos personales. Busca dos cosas principales:
- Dar a las personas **más control** sobre cómo se usan sus datos personales.
- Obligar a las organizaciones a aplicar principios claros de tratamiento (minimización, limitación de finalidad, seguridad, transparencia, etc.).

Algunas ideas clave:

- **Dato personal**: cualquier información que identifique o pueda identificar a una persona (nombre, correo, documento, IP, etc.).
- **Dato sensible**: categorías especiales como salud, orientación sexual, opiniones políticas, religión, etc.
- **Minimización de datos**: solo recolectar y procesar lo estrictamente necesario para la finalidad declarada.
- **Seguridad**: aplicar medidas técnicas y organizativas adecuadas (cifrado, control de acceso, anonimización, pseudonimización, auditoría, etc.).

> En este notebook, usaremos NER como **una herramienta técnica** que puede ayudarnos a localizar y anonimizar datos personales en textos antes de enviarlos a modelos de lenguaje o pipelines más complejos.

### 2.2. Colombia – Ley 1581 de 2012

En Colombia, la **Ley 1581 de 2012** establece disposiciones generales para la protección de datos personales.

Puntos básicos para este módulo:

- Reconoce el derecho de todas las personas a conocer, actualizar y rectificar sus datos.
- Define principios como **legalidad, finalidad, libertad, veracidad, transparencia, acceso y circulación restringida, seguridad y confidencialidad**.
- Exige a las organizaciones adoptar medidas de seguridad y políticas internas de tratamiento de datos.

### 2.3. México – LFPDPPP

En México, la referencia clave es la **Ley Federal de Protección de Datos Personales en Posesión de los Particulares (LFPDPPP)**.

Elementos a tener presentes:

- Aplica al sector privado que trata datos personales.
- Se basa en principios como **consentimiento**, **finalidad**, **proporcionalidad**, **calidad de los datos** y **responsabilidad**.

### 2.4. Perú – Ley 29733

En Perú, la **Ley 29733** regula el tratamiento de datos personales en bancos de datos, tanto públicos como privados.


Aspectos clave:

- Establece principios como **legalidad, consentimiento, proporcionalidad, calidad, seguridad y disposición de recurso**.

- Crea la Autoridad Nacional de Protección de Datos Personales como ente supervisor.


### 2.5. Conexión con NER y LLM

En todos estos marcos regulatorios hay ideas comunes:

- Proteger datos personales.
- Limitar el uso de esos datos a finalidades legítimas.
- Implementar medidas de seguridad para reducir riesgos.

En las siguientes secciones usaremos NER en español como **mecanismo de apoyo** para:

- Detectar datos personales en texto.
- Anonimizar o pseudonimizar antes de enviar información a un LLM.
- Discutir dónde NER ayuda y dónde no es suficiente.

<a id="sec-recordatorio-tecnico"></a>
## 3. Recordatorio técnico: Token classification y NER

La tarea de **token classification** asigna una etiqueta a algunos tokens dentro de un texto.  
NER (Named Entity Recognition) es un caso particular donde esas etiquetas indican entidades como:

- Personas (PER)
- Organizaciones (ORG)
- Lugares (LOC)
- Otros tipos según el modelo (MISC, enfermedades, medicamentos, artículos legales, etc.)

En Hugging Face, usamos normalmente:

- El método `pipeline("ner")` para inferencia rápida.
- El parámetro `aggregation_strategy="simple"` para agrupar tokens contiguos en entidades completas (por ejemplo, unir “Jorge” + “Ignacio”+ "Blanco").

Diagrama conceptual:

1. Texto de entrada (string en español).  
2. Tokenización y paso por el modelo NER.  
3. Salida por token (B-PER, I-PER, B-ORG, etc.).  
4. Agregación en entidades:  
   - `entity_group`: PER, ORG, LOC, etc.  
   - `word`: texto de la entidad.  
   - `start`, `end`: posiciones de la entidad en el texto.  
   - `score`: confianza del modelo.

En las secciones siguientes haremos:

- Carga de un modelo NER general en español.
- Experimentos con textos que contienen datos personales.
- Anonimización básica usando las entidades detectadas.

<a id="sec-modelo-ner"></a>
## 4. Selección de modelo NER en español

Trabajaremos con modelos de Hugging Face ya entrenados para NER en español.

### 4.1. Catálogo de modelos sugerido

Busquen y revisen lo modelos NER en español en:

- [https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner](https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner)

Verás modelos:

- Generales (textos noticiosos, CoNLL-2002 en español).
- Multilingües (por ejemplo XLM-R).
- Específicos de dominios como salud o legal.

### 4.2. Modelo base recomendado para empezar

En este notebook utilizaremos como modelo base:

- `MMG/xlm-roberta-large-ner-spanish` – modelo NER en español entrenado sobre la porción española de CoNLL-2002.

En una celda de código, pueden cargarlo así:

```python
from transformers import pipeline

modelo_base = "MMG/xlm-roberta-large-ner-spanish"

ner_es = pipeline(
    "ner",
    model=modelo_base,
    aggregation_strategy="simple"
)

texto_demo = "Mi nombre es Abel Cierto, vivo en Perú y soy estudiante de BSG Institute."
# luego en una lista carguen el nombre de los integrantes del gurpo, donde viven y en donde trabajan
ner_es(texto_demo)
```

A partir de aquí, `ner_es` será nuestro pipeline principal para los primeros ejercicios.

In [2]:
from transformers import pipeline

modelo_base = "MMG/xlm-roberta-large-ner-spanish"

ner_es = pipeline(
    "ner",
    model=modelo_base,
    aggregation_strategy="simple"
)

text_demo = "Mi nombre es Abel Cierto, vivo en Perú y soy ingeniero de software y soy estudiante de BSG Institute"

ner_es(text_demo)

Device set to use cpu


[{'entity_group': 'PER',
  'score': np.float32(0.9980915),
  'word': 'Abel Cierto',
  'start': 13,
  'end': 24},
 {'entity_group': 'LOC',
  'score': np.float32(0.85963047),
  'word': 'Perú',
  'start': 34,
  'end': 38},
 {'entity_group': 'ORG',
  'score': np.float32(0.9928886),
  'word': 'BSG Institute',
  'start': 87,
  'end': 100}]

<a id="sec-ejercicio-1"></a>
## 5. Ejercicio 1 – NER básico en español


**Objetivo:**  
Comprobar que el modelo NER en español detecta correctamente personas, organizaciones y lugares en frases simples, y relacionar cada entidad con la idea de “dato personal”.

### 5.1. Código base

Usa algo como lo siguiente en una celda de código:

```python
texto_1 = "aqui coloquen el nombre de los integrantes del grupo + la otra informacion pedida en el ejercicio anterior"
resultado_1 = ner_es(texto_1)
resultado_1
```

In [11]:
text_1 = "Mi equipo de desarrollo de software está compuesto por Branco Villegas, Dominik Mendoza y Abel Cierto. Todos vivimos en Perú y trabajamos para MedaluTech como ingenieros de software y estudiantes de BSG Institute."
result_1 = ner_es(text_1)
result_1

[{'entity_group': 'PER',
  'score': np.float32(0.9991867),
  'word': 'Branco Villegas',
  'start': 55,
  'end': 70},
 {'entity_group': 'PER',
  'score': np.float32(0.89694196),
  'word': 'Dominik Mendoza',
  'start': 72,
  'end': 87},
 {'entity_group': 'PER',
  'score': np.float32(0.99916893),
  'word': 'Abel Cierto',
  'start': 90,
  'end': 101},
 {'entity_group': 'LOC',
  'score': np.float32(0.74800205),
  'word': 'Perú',
  'start': 120,
  'end': 124},
 {'entity_group': 'ORG',
  'score': np.float32(0.9921322),
  'word': 'MedaluTech',
  'start': 143,
  'end': 153},
 {'entity_group': 'ORG',
  'score': np.float32(0.9934223),
  'word': 'BSG Institute',
  'start': 199,
  'end': 212}]

### 5.2. Preguntas a resolver

1. ¿Qué entidades aparecen en la salida (campo `entity_group`)?  
**Respuesta:** Aparecen tres tipos de entidades: **PER** (persona), **LOC** (lugar) y **ORG** (organización).

2. ¿Qué texto exacto detectó el modelo para cada entidad (`word`)?

| entity_group | word | score |
|---|---|---|
| PER | Branco Villegas | 0.999 |
| PER | Dominik Mendoza | 0.897 |
| PER | Abel Cierto | 0.999 |
| LOC | Perú | 0.748 |
| ORG | MedaluTech | 0.992 |
| ORG | BSG Institute | 0.993 |

3. ¿Cuáles de esas entidades son claramente **datos personales** (PII)?  
Las tres entidades **PER** (Branco Villegas, Dominik Mendoza, Abel Cierto) son claramente PII, ya que identifican de forma directa a personas naturales. La entidad **LOC** ("Perú") por sí sola es un dato de baja granularidad (país), pero combinada con los nombres de las personas se convierte en información contextual sobre su residencia, lo cual también aporta al perfil de PII. Las entidades **ORG** ("MedaluTech", "BSG Institute") no son PII en sí mismas, pero al asociarse con los nombres detectados revelan el lugar de trabajo/estudio de personas identificables, lo cual es un dato personal indirecto (dato laboral vinculado a un individuo).

4. ¿Qué implicaciones tendría enviar este texto a un LLM externo sin ningún tipo de anonimización?  
Se estarían transmitiendo a un tercero (el proveedor del LLM) datos que permiten identificar plenamente a tres personas (nombre completo), su país de residencia y su relación laboral/académica con dos organizaciones específicas. Esto implica:

- **Riesgo de exposición de PII** fuera del control de la organización, sin garantías sobre almacenamiento, retención o reutilización de esos datos por parte del proveedor externo.
- **Incumplimiento del principio de minimización de datos** (GDPR y Ley 1581 de 2012 en Colombia, LFPDPPP en México, Ley 29733 en Perú), ya que se envía más información personal de la estrictamente necesaria para la tarea.
- **Falta de trazabilidad y consentimiento**: las personas mencionadas no han autorizado explícitamente que sus datos sean procesados por un servicio de IA externo.
- **Riesgo reputacional y legal** para la organización si el LLM externo no cumple con estándares de seguridad adecuados o si los datos se filtran o se usan para reentrenar modelos.

> Nota: este ejercicio sirve para fijar la idea de que NER puede ser la “primera capa” de detección de datos personales en un pipeline de IA.

<a id="sec-ejercicio-2"></a>
## 6. Ejercicio 2 – Anonimización básica con NER

**Objetivo:**  
Usar la salida de NER para generar una versión anonimizada del texto, sustituyendo las entidades por etiquetas genéricas. Esto se conecta directamente con los principios de minimización y protección de datos de GDPR y de las leyes locales.

### 6.1. Función de anonimización

En una celda de código, define una función como esta:

```python
def anonimizar_entidades(texto, entidades):
    # Ordenar las entidades de atrás hacia adelante para no romper índices
    entidades_ordenadas = sorted(entidades, key=lambda x: x["start"], reverse=True)
    for ent in entidades_ordenadas:
        etiqueta = ent["entity_group"]
        inicio = ent["start"]
        fin = ent["end"]
        texto = texto[:inicio] + f"[{etiqueta}]" + texto[fin:]
    return texto

texto_anon = anonimizar_entidades(texto_1, resultado_1)

print("Texto original:")
print(texto_1)
print("\nTexto anonimizado:")
print(texto_anon)
```

In [12]:
def anonimizar_entidades(texto, entidades):
    # Ordenar las entidades de atrás hacia adelante para no romper índices
    entidades_ordenadas = sorted(entidades, key=lambda x: x["start"], reverse=True)
    for ent in entidades_ordenadas:
        etiqueta = ent["entity_group"]
        inicio = ent["start"]
        fin = ent["end"]
        texto = texto[:inicio] + f"[{etiqueta}]" + texto[fin:]
    return texto

texto_anon = anonimizar_entidades(text_1, result_1)

print("Texto original:")
print(text_1)
print("\nTexto anonimizado:")
print(texto_anon)

Texto original:
Mi equipo de desarrollo de software está compuesto por Branco Villegas, Dominik Mendoza y Abel Cierto. Todos vivimos en Perú y trabajamos para MedaluTech como ingenieros de software y estudiantes de BSG Institute.

Texto anonimizado:
Mi equipo de desarrollo de software está compuesto por [PER], [PER] y [PER]. Todos vivimos en [LOC] y trabajamos para [ORG] como ingenieros de software y estudiantes de [ORG].


### 6.2. Preguntas de reflexión

- **¿El texto anonimizado sigue siendo útil para tareas de análisis semántico?**  
Sí, en buena medida. La estructura sintáctica y el sentido general de la frase se conservan: sigue siendo claro que hay un equipo de tres personas que vive en un país, trabaja para una organización y estudia en otra. Para tareas como clasificación de texto, detección de intención o resumen, el texto anonimizado sigue siendo útil, aunque se pierde la capacidad de responder preguntas específicas sobre "quién" o "dónde" exactamente.

- **¿Crees que un LLM podría entender igual el contexto con `[PER]`, `[ORG]` y `[LOC]`?**  
Un LLM puede entender el contexto general (que hay personas, una ubicación y organizaciones involucradas), pero pierde matices importantes: no puede diferenciar entre las tres personas mencionadas (todas quedan como `[PER]` genérico), ni distinguir que una organización es el empleador y otra la institución educativa, ya que ambas quedan como `[ORG]`. Esto puede afectar tareas donde el modelo necesita razonar sobre relaciones específicas entre entidades (ej. "¿en qué institución estudian, distinto de dónde trabajan?").

- **¿Qué pasa si el modelo NER no detecta alguna entidad importante (por ejemplo, un número de documento o un correo)?**  
Si el modelo NER no detecta esa entidad (porque no fue entrenado para reconocer ese tipo de dato, como números de documento, correos electrónicos, teléfonos o direcciones IP), esa información **queda expuesta sin anonimizar** en el texto final. Esto es un riesgo real: el modelo `MMG/xlm-roberta-large-ner-spanish` solo reconoce PER, LOC, ORG y MISC, por lo que datos sensibles como cédulas, correos o números de historia clínica pasarían desapercibidos. Por eso la anonimización basada únicamente en NER es insuficiente para un flujo de cumplimiento real: se debe complementar con **expresiones regulares** (para correos, teléfonos, documentos con formato conocido) y **revisión humana** antes de enviar el texto a un tercero.

> En un flujo de cumplimiento real, la anonimización basada en NER se complementa con reglas adicionales y revisión humana.

<a id="sec-ejercicio-3"></a>
## 7. Ejercicio 3 – Frases con mayor complejidad

**Objetivo:**  
Ver cómo se comporta el modelo NER en frases más realistas, con combinaciones de personas, organizaciones, lugares y conceptos propios de salud, legal o finanzas.

### 7.1. Conjunto de frases de prueba

In [13]:
textos = [
    "Carlos trabaja en Microsoft y vive en Madrid.",
    "La paciente Ana Torres fue atendida en la Clínica San Rafael de Lima.",
    "El abogado Juan Pérez firmó el contrato en Ciudad de México para Inversiones Andinas S.A.",
    "El número de historia clínica de María Gómez fue registrado en el hospital central.",
]

for i, t in enumerate(textos, 1):
    print(f"\nTexto {i}")
    print(t)
    print(ner_es(t))


Texto 1
Carlos trabaja en Microsoft y vive en Madrid.
[{'entity_group': 'PER', 'score': np.float32(0.98660916), 'word': 'Carlos', 'start': 0, 'end': 6}, {'entity_group': 'ORG', 'score': np.float32(0.9893127), 'word': 'Microsoft', 'start': 18, 'end': 27}, {'entity_group': 'LOC', 'score': np.float32(0.9996655), 'word': 'Madrid', 'start': 38, 'end': 44}]

Texto 2
La paciente Ana Torres fue atendida en la Clínica San Rafael de Lima.
[{'entity_group': 'PER', 'score': np.float32(0.9974449), 'word': 'Ana Torres', 'start': 12, 'end': 22}, {'entity_group': 'LOC', 'score': np.float32(0.9861086), 'word': 'C', 'start': 42, 'end': 43}, {'entity_group': 'LOC', 'score': np.float32(0.9728108), 'word': 'lí', 'start': 43, 'end': 45}, {'entity_group': 'LOC', 'score': np.float32(0.8366556), 'word': 'nica San Rafael', 'start': 45, 'end': 60}, {'entity_group': 'LOC', 'score': np.float32(0.98467815), 'word': 'Lima', 'start': 64, 'end': 68}]

Texto 3
El abogado Juan Pérez firmó el contrato en Ciudad de Méxic

### 7.2. Preguntas para el estudiante

- **¿En cuáles textos el modelo funciona bien?**  
El **Texto 1** ("Carlos trabaja en Microsoft y vive en Madrid") y el **Texto 3** ("El abogado Juan Pérez firmó el contrato en Ciudad de México para Inversiones Andinas S.A.") funcionan muy bien: el modelo detecta correctamente y con scores altos (>0.97) a la persona (PER), la organización (ORG) y el lugar (LOC), agrupando los tokens de forma coherente (por ejemplo, "Ciudad de México" e "Inversiones Andinas S.A" quedan como entidades completas y bien delimitadas).

- **¿En qué casos empieza a fallar (entidades que no detecta o que clasifica mal)?**  
El **Texto 2** muestra una falla clara de segmentación: el nombre "Clínica San Rafael" se fragmenta en tres entidades LOC distintas ("C", "lí", "nica San Rafael") en lugar de agruparse como una sola entidad ORG (u otra etiqueta que la identifique como institución de salud). Esto evidencia un problema de tokenización/agregación del modelo con nombres propios compuestos que contienen palabras poco frecuentes en el corpus de entrenamiento (CoNLL-2002 es de dominio noticioso, no clínico).

El **Texto 4** falla por omisión: el modelo solo detecta a "María Gómez" (PER), pero **no reconoce "hospital central" como ORG/LOC**, ni mucho menos el concepto "número de historia clínica" como un dato sensible. Esto confirma que el modelo general no tiene noción de entidades clínicas.

- **¿Qué tipos de datos importantes para cumplimiento no aparecen como entidades (ej. "número de historia clínica")?**  
El modelo no reconoce:
  - Números de historia clínica, cédulas, pasaportes o cualquier identificador numérico de una persona.
  - Diagnósticos, medicamentos o procedimientos médicos (dominio biomédico).
  - Fechas exactas, números de expediente o de contrato (dominio legal).
  - Correos electrónicos, teléfonos, direcciones IP.
  - Nombres de instituciones con estructura poco común (como se vio con "Clínica San Rafael"), que terminan mal segmentadas o mal etiquetadas.

- **¿Qué riesgos ocurren si confiamos solo en NER para anonimización?**  
  - **Falsos negativos silenciosos**: datos sensibles como el "número de historia clínica" o un correo pueden quedar completamente expuestos porque el modelo simplemente no los reconoce como entidad.
  - **Fragmentación incorrecta de entidades** (como en "Clínica San Rafael"): si se anonimiza solo parte del nombre de una institución, el resto del texto puede seguir siendo identificable o generar anonimizaciones inconsistentes.
  - **Falsa sensación de seguridad**: un equipo de cumplimiento podría asumir que el texto ya está "limpio" después de aplicar NER, cuando en realidad persisten datos personales o sensibles no detectados.
  - **Dependencia del dominio del modelo**: un modelo entrenado en noticias generales (CoNLL-2002) no está preparado para textos clínicos o legales, por lo que se requieren modelos especializados (biomédicos, legales) o reglas complementarias (regex, listas de términos, revisión humana) para lograr una anonimización robusta y alineada con GDPR y las leyes locales de protección de datos.

<a id="sec-ejercicio-4"></a>
## Ejercicio 4 – Rellenar máscaras con `fill-mask` (Masked Language Modeling)

Además de NER, otra tarea útil para hablar de **sesgos** y **riesgos** en LLM es el relleno de máscaras (*fill-mask*). En esta tarea, el modelo recibe una frase con una palabra oculta y debe predecir qué palabra encaja mejor en ese contexto.

En Hugging Face, esta tarea se usa con el pipeline:

- `pipeline("fill-mask")` para inglés (por defecto o indicando modelo).
- Para español, podemos usar un modelo de lenguaje entrenado con objetivo de *masked language modeling* adecuado al idioma.

> Importante: el token de máscara debe coincidir con el del modelo (por ejemplo, BERT usa `[MASK]`, RoBERTa usa `<mask>`).

### 4.1. Ejemplo básico en español

En una celda de código:

In [16]:
from transformers import pipeline

# Modelo fill-mask multilingüe (ejemplo, puedes cambiarlo por otro apropiado)
fill_mask = pipeline(
    "fill-mask",
    model="dccuchile/bert-base-spanish-wwm-cased"  # modelo BERT en español
)

texto_mask = "Mi nombre es [MASK] y trabajo en una empresa de tecnología."
fill_mask(texto_mask)

Device set to use cpu


[{'score': 0.35630524158477783,
  'token': 3,
  'token_str': '[UNK]',
  'sequence': 'Mi nombre es y trabajo en una empresa de tecnología.'},
 {'score': 0.010257055051624775,
  'token': 6182,
  'token_str': 'Alex',
  'sequence': 'Mi nombre es Alex y trabajo en una empresa de tecnología.'},
 {'score': 0.009632506407797337,
  'token': 26774,
  'token_str': 'Samantha',
  'sequence': 'Mi nombre es Samantha y trabajo en una empresa de tecnología.'},
 {'score': 0.009423261508345604,
  'token': 5204,
  'token_str': 'David',
  'sequence': 'Mi nombre es David y trabajo en una empresa de tecnología.'},
 {'score': 0.007947920821607113,
  'token': 16726,
  'token_str': 'Chloe',
  'sequence': 'Mi nombre es Chloe y trabajo en una empresa de tecnología.'}]

### 4.2. Observando sesgos en predicciones

Prueben ahora con frases que puedan revelar estereotipos:

In [17]:
oraciones = [
    "El doctor [MASK] atendió a la paciente.",
    "La enfermera [MASK] cuidó al paciente.",
    "El ingeniero [MASK] diseñó el puente.",
    "La secretaria [MASK] organizó la reunión.",
]

for t in oraciones:
    print(f"\nOración: {t}")
    for pred in fill_mask(t):
        print(f"  {pred['sequence']}  (score={pred['score']:.3f})")
    # Limitar a las 5 mejores predicciones por defecto


Oración: El doctor [MASK] atendió a la paciente.
  El doctor atendió a la paciente.  (score=0.286)
  El doctor no atendió a la paciente.  (score=0.097)
  El doctor que atendió a la paciente.  (score=0.092)
  El doctor también atendió a la paciente.  (score=0.023)
  El doctor ya atendió a la paciente.  (score=0.010)

Oración: La enfermera [MASK] cuidó al paciente.
  La enfermera que cuidó al paciente.  (score=0.319)
  La enfermera cuidó al paciente.  (score=0.117)
  La enfermera también cuidó al paciente.  (score=0.034)
  La enfermera no cuidó al paciente.  (score=0.029)
  La enfermera encargada cuidó al paciente.  (score=0.017)

Oración: El ingeniero [MASK] diseñó el puente.
  El ingeniero que diseñó el puente.  (score=0.453)
  El ingeniero diseñó el puente.  (score=0.101)
  El ingeniero también diseñó el puente.  (score=0.042)
  El ingeniero quien diseñó el puente.  (score=0.032)
  El ingeniero principal diseñó el puente.  (score=0.024)

Oración: La secretaria [MASK] organizó la reun

### 4.3. Preguntas de reflexión

- **¿Qué nombres o palabras suele proponer el modelo para cada profesión?**  
En el primer ejemplo ("Mi nombre es [MASK]..."), el modelo propone principalmente el token `[UNK]` (desconocido, con el score más alto), seguido de nombres propios como *Alex*, *Samantha*, *David* y *Chloe* — una mezcla de nombres anglosajones/neutros más que hispanos, lo cual llama la atención dado que el texto está en español.

En las oraciones con `[MASK]` entre el sustantivo de la profesión y el verbo, el modelo **no propone nombres propios en absoluto**: en su lugar completa con conectores o adjetivos genéricos (nada, "que", "no", "también", "ya") o con calificativos del cargo ("general", "técnica", "ejecutiva", "principal", "encargada"). Es decir, el modelo interpretó el hueco como una posición sintáctica (pronombre relativo/adverbio) en vez de un nombre propio de persona.

- **¿Detectan estereotipos de género, cultura o rol profesional en las predicciones?**  
Sí, aunque de forma indirecta por cómo se resolvió el `[MASK]`:
  - Para **"secretaria"** el modelo propone calificativos como "general", "técnica", "ejecutiva" — términos que refuerzan un rol administrativo/de apoyo, consistente con el estereotipo de la secretaria como cargo subordinado o de coordinación, no de autoridad técnica.
  - Para **"ingeniero"** y **"doctor"**, las completions son mayormente neutras ("que", "no", "también"), sin adjetivos que califiquen jerarquía o especialización, lo que sugiere que el modelo "necesita" menos matizar estas profesiones, tratándolas como más "completas" o autosuficientes por defecto.
  - En el primer ejercicio, la ausencia de nombres hispanos comunes (María, Carlos, Juan) y la preferencia por nombres extranjeros (Alex, Samantha, David, Chloe) para un contexto en español sugiere un sesgo cultural derivado del corpus de entrenamiento, que probablemente contiene más menciones de nombres en inglés en contextos tecnológicos ("trabajo en una empresa de tecnología").

- **¿Qué implicaciones tiene esto para aplicaciones reales (por ejemplo, asistentes de carrera, generación de ejemplos en educación, etc.)?**  
Si estos modelos se usan para generar ejemplos automáticos en materiales educativos, chatbots de orientación vocacional o generación de contenido de RR.HH., podrían:
  - Reforzar asociaciones sesgadas entre género/rol (ej. mujeres en roles de "apoyo" como secretaria y hombres/neutros en roles técnicos).
  - Invisibilizar la diversidad cultural local al preferir nombres extranjeros sobre nombres hispanos en contextos donde el idioma y la audiencia son claramente hispanohablantes.
  - Generar ejemplos poco representativos que, replicados a escala, normalicen estereotipos sin que el usuario final se dé cuenta de que provienen de un sesgo estadístico del modelo y no de una elección deliberada.

- **¿Cómo conectarían esta observación con las obligaciones de no discriminación y trato justo en marcos como GDPR o las leyes locales?**  
Aunque el GDPR y las leyes locales (Ley 1581 en Colombia, LFPDPPP en México, Ley 29733 en Perú) se centran principalmente en la protección de **datos personales**, sus principios de **transparencia**, **trato justo** y **responsabilidad** en el tratamiento de información también aplican, por extensión, a los sistemas de IA que procesan o generan contenido sobre personas. Un sistema que perpetúa sesgos de género o culturales en la generación de texto puede:
  - Vulnerar principios de **no discriminación** reconocidos en marcos más amplios de ética en IA (como los lineamientos de la UE para IA confiable), que exigen evaluar y mitigar sesgos algorítmicos antes de desplegar modelos en producción.
  - Generar responsabilidad organizacional si el sesgo del modelo afecta decisiones reales (por ejemplo, en generación automática de descripciones de puestos o recomendaciones de carrera), conectándose con la exigencia de **auditorías y evaluaciones de impacto** que varios marcos regulatorios de IA (como el AI Act europeo) ya contemplan como complemento al GDPR.

> Este ejercicio complementa NER: aquí no buscamos identificar entidades, sino entender **cómo el modelo “imagina” el mundo** cuando tiene que completar una frase. Eso nos da material para discutir sesgos y riesgos éticos.

<a id="sec-ejercicio-5"></a>
## 8. Ejercicio 5 – NER aplicado a salud / medicina

El dominio de salud es especialmente muy sensible:

- GDPR lo trata como **categoría especial de dato** (salud).
- Las leyes locales suelen exigir medidas reforzadas para historias clínicas, diagnósticos, medicamentos, etc.

### 8.1. Texto de ejemplo clínico

In [18]:
texto_salud = "La paciente Laura Medina recibió tratamiento con paracetamol en el Hospital Central de Bogotá."
resultado_salud = ner_es(texto_salud)
resultado_salud

[{'entity_group': 'PER',
  'score': np.float32(0.98469496),
  'word': 'Laura Medina',
  'start': 12,
  'end': 24},
 {'entity_group': 'LOC',
  'score': np.float32(0.9772317),
  'word': 'Hospital Central de Bogotá',
  'start': 67,
  'end': 93}]

Probablemente el modelo general detectará personas, hospitales, ciudades… pero **no** etiquetas específicas como “fármaco” o “diagnóstico”.

### 8.2. Recursos de modelos y datasets clínicos

Para ir más allá del modelo general, sugiero explorar:

- Repositorio de modelos biomédicos y clínicos en español:  
  - GitHub: https://github.com/PlanTL-GOB-ES/lm-biomedical-clinical-es  
- Datasets en Hugging Face:  
  - **CANTEMIST NER**: [https://huggingface.co/datasets/PlanTL-GOB-ES/cantemist-ner](https://huggingface.co/datasets/PlanTL-GOB-ES/cantemist-ner)  
  - **PharmaCoNER**: [https://huggingface.co/datasets/PlanTL-GOB-ES/pharmaconer](https://huggingface.co/datasets/PlanTL-GOB-ES/pharmaconer)

### 8.3. Actividad

1. Ejecuten el ejemplo clínico con el modelo general.  
2. Investigen un modelo NER biomédico en español en Hugging Face.  
3. Comparen los resultados de NER general vs. NER biomédico.  
4. Identifiquen qué entidades clínicas importantes aparecen solo con el modelo especializado.

In [19]:
from transformers import pipeline

# Modelo NER biomédico/clínico en español (fine-tuned para NER, dominio farmacológico)
modelo_biomedico = "lcampillos/roberta-es-clinical-trials-ner"

ner_bio = pipeline(
    "ner",
    model=modelo_biomedico,
    aggregation_strategy="simple"
)

print("=== Resultado con modelo NER general (MMG/xlm-roberta-large-ner-spanish) ===")
print(resultado_salud)

print("\n=== Resultado con modelo NER biomédico ===")
resultado_salud_bio = ner_bio(texto_salud)
print(resultado_salud_bio)

C:\Users\abel.cierto\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abel.cierto\.cache\huggingface\hub\models--lcampillos--roberta-es-clinical-trials-ner. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Device set to use cpu


=== Resultado con modelo NER general (MMG/xlm-roberta-large-ner-spanish) ===
[{'entity_group': 'PER', 'score': np.float32(0.98469496), 'word': 'Laura Medina', 'start': 12, 'end': 24}, {'entity_group': 'LOC', 'score': np.float32(0.9772317), 'word': 'Hospital Central de Bogotá', 'start': 67, 'end': 93}]

=== Resultado con modelo NER biomédico ===
[{'entity_group': 'PROC', 'score': np.float32(0.9990583), 'word': ' tratamiento', 'start': 33, 'end': 44}, {'entity_group': 'CHEM', 'score': np.float32(0.9976332), 'word': ' paracetamol', 'start': 49, 'end': 60}]


### 8.4. Análisis comparativo: NER general vs. NER biomédico

**Texto de prueba:** "La paciente Laura Medina recibió tratamiento con paracetamol en el Hospital Central de Bogotá."

**1. Resultado con el modelo general (`MMG/xlm-roberta-large-ner-spanish`):**
```
[{'entity_group': 'PER', 'word': 'Laura Medina', 'score': 0.985},
 {'entity_group': 'LOC', 'word': 'Hospital Central de Bogotá', 'score': 0.977}]
```
El modelo detecta correctamente a la paciente (`PER`) y agrupa muy bien "Hospital Central de Bogotá" como una sola entidad `LOC` (a diferencia del caso de "Clínica San Rafael" del Ejercicio 3, aquí no hubo fragmentación). Sin embargo, **no detecta "tratamiento" ni "paracetamol"** — para este modelo son solo palabras comunes del texto, ya que fue entrenado en CoNLL-2002 (dominio noticioso, sin categorías clínicas).

**2. Resultado con el modelo NER biomédico (`lcampillos/roberta-es-clinical-trials-ner`):**
```
[{'entity_group': 'PROC', 'word': 'tratamiento', 'score': 0.999},
 {'entity_group': 'CHEM', 'word': 'paracetamol', 'score': 0.998}]
```
Este modelo, entrenado sobre ensayos clínicos en español, detecta con scores muy altos (>0.99) dos entidades completamente invisibles para el modelo general:
- **"tratamiento"** como `PROC` (procedimiento clínico).
- **"paracetamol"** como `CHEM` (sustancia química/fármaco).

En cambio, **no reconoce a "Laura Medina" ni a "Hospital Central de Bogotá"**, porque su esquema de etiquetas está enfocado en conceptos clínicos (fármacos, procedimientos, enfermedades) y no en entidades generales como personas o lugares.

**3. Comparación de resultados:**

| Entidad | Modelo general | Modelo biomédico |
|---|---|---|
| Laura Medina (persona) | ✅ PER (0.985) | ❌ No detectado |
| Hospital Central de Bogotá (lugar) | ✅ LOC (0.977), bien agrupado | ❌ No detectado |
| tratamiento (procedimiento) | ❌ No detectado | ✅ PROC (0.999) |
| paracetamol (fármaco) | ❌ No detectado | ✅ CHEM (0.998) |

**4. Entidades clínicas que solo aparecen con el modelo especializado:**
- **"paracetamol"** (CHEM) y **"tratamiento"** (PROC) son justo el tipo de información que el modelo general ignora por completo. Esto confirma que **cada modelo "ve" un subconjunto distinto y complementario de la realidad del texto**: ninguno de los dos, por sí solo, cubre todas las entidades relevantes.

**5. Conclusión para cumplimiento:**
- Los dos modelos son **complementarios, no sustitutos**: el general aporta identificación de personas y lugares (PII clásica), y el biomédico aporta fármacos y procedimientos (datos de salud, categoría especial bajo GDPR).
- Un pipeline de anonimización real en el dominio clínico **debe ejecutar ambos modelos sobre el mismo texto** y fusionar sus resultados (cuidando los offsets `start`/`end` para no generar solapamientos al anonimizar), ya que omitir cualquiera de los dos deja expuesto un tipo distinto de dato sensible.
- Ninguno de los dos modelos detecta identificadores numéricos (historia clínica, DNI), por lo que se requieren además reglas complementarias (regex) para un cumplimiento robusto con GDPR, Ley 1581 (Colombia), LFPDPPP (México) y Ley 29733 (Perú).
- Este resultado es la evidencia empírica más clara del notebook de que la **especialización de dominio** en NER no es opcional en salud: sin el modelo biomédico, "paracetamol" habría quedado completamente expuesto en cualquier texto enviado a un LLM externo.

<a id="sec-ejercicio-6"></a>
## 9. Ejercicio 6 – NER aplicado a documentos legales

En el dominio legal, los textos contienen:

- Nombres de personas y empresas.
- Juzgados, ciudades, fechas, números de contrato.
- Referencias a artículos de leyes, resoluciones, sentencias.

Todo esto es relevante para **cumplimiento** y para sistemas que revisan contratos, demandas o políticas.

### 9.1. Texto de ejemplo legal

In [20]:
texto_legal = (
    "El abogado Carlos Ramírez presentó la demanda ante el Juzgado Primero de Lima "
    "el 12 de mayo de 2026 en representación de Grupo Andes S.A.S"
)

resultado_legal = ner_es(texto_legal)
resultado_legal

[{'entity_group': 'PER',
  'score': np.float32(0.9977687),
  'word': 'Carlos Ramírez',
  'start': 11,
  'end': 25},
 {'entity_group': 'ORG',
  'score': np.float32(0.93452877),
  'word': 'Ju',
  'start': 54,
  'end': 56},
 {'entity_group': 'ORG',
  'score': np.float32(0.869379),
  'word': 'z',
  'start': 56,
  'end': 57},
 {'entity_group': 'ORG',
  'score': np.float32(0.8725548),
  'word': 'gado Primero de',
  'start': 57,
  'end': 72},
 {'entity_group': 'LOC',
  'score': np.float32(0.4729005),
  'word': 'Lima',
  'start': 73,
  'end': 77},
 {'entity_group': 'ORG',
  'score': np.float32(0.99749076),
  'word': 'Grupo Andes S',
  'start': 121,
  'end': 134},
 {'entity_group': 'ORG',
  'score': np.float32(0.99047107),
  'word': 'A.S',
  'start': 135,
  'end': 138}]

### 9.2. Recursos legales en Hugging Face

- Modelo NER legal en español (ejemplo):  
  - `agomez302/nlp-dr-ner`  
  - Página: [https://huggingface.co/agomez302/nlp-dr-ner](https://huggingface.co/agomez302/nlp-dr-ner)
- Guía de fine-tuning de modelos de clasificación de tokens para datos legales:  
  - [https://huggingface.co/blog/bikashpatra/legal-data-token-classification-fine-tuning](https://huggingface.co/blog/bikashpatra/legal-data-token-classification-fine-tuning)

In [22]:
print("=== Texto legal ===")
print(texto_legal)
print("\n=== Resultado NER general (MMG/xlm-roberta-large-ner-spanish) ===")
print(resultado_legal)

# Intento con modelo NER legal especializado
from transformers import pipeline

try:
    modelo_legal = "agomez302/nlp-dr-ner"
    ner_legal = pipeline(
        "ner",
        model=modelo_legal,
        aggregation_strategy="simple"
    )
    resultado_legal_esp = ner_legal(texto_legal)
    print("\n=== Resultado NER legal especializado ===")
    print(resultado_legal_esp)
except Exception as e:
    print(f"\nNo se pudo cargar el modelo legal especializado: {e}")

=== Texto legal ===
El abogado Carlos Ramírez presentó la demanda ante el Juzgado Primero de Lima el 12 de mayo de 2026 en representación de Grupo Andes S.A.S

=== Resultado NER general (MMG/xlm-roberta-large-ner-spanish) ===
[{'entity_group': 'PER', 'score': np.float32(0.9977687), 'word': 'Carlos Ramírez', 'start': 11, 'end': 25}, {'entity_group': 'ORG', 'score': np.float32(0.93452877), 'word': 'Ju', 'start': 54, 'end': 56}, {'entity_group': 'ORG', 'score': np.float32(0.869379), 'word': 'z', 'start': 56, 'end': 57}, {'entity_group': 'ORG', 'score': np.float32(0.8725548), 'word': 'gado Primero de', 'start': 57, 'end': 72}, {'entity_group': 'LOC', 'score': np.float32(0.4729005), 'word': 'Lima', 'start': 73, 'end': 77}, {'entity_group': 'ORG', 'score': np.float32(0.99749076), 'word': 'Grupo Andes S', 'start': 121, 'end': 134}, {'entity_group': 'ORG', 'score': np.float32(0.99047107), 'word': 'A.S', 'start': 135, 'end': 138}]


Device set to use cpu



=== Resultado NER legal especializado ===
[{'entity_group': 'DATE', 'score': np.float32(0.9994429), 'word': '12 de mayo de 2026', 'start': 81, 'end': 99}]


### 9.3. Actividad

- A partir del resultado de NER general:
  - ¿Qué entidades son importantes para un flujo de revisión legal?  
  - ¿Qué entidades legales relevantes no aparecen (ej. tipo de proceso, número de expediente, referencia a norma)?  
- Discutan y registren entre ustedes qué necesitaría un sistema NER “serio” para legal:
  - Etiquetas más específicas.  
  - Entrenamiento con datasets legales.  
  - Reglas de negocio encima del modelo.

### 9.4. Análisis de la Actividad (resultados reales)

**Texto de prueba:** "El abogado Carlos Ramírez presentó la demanda ante el Juzgado Primero de Lima el 12 de mayo de 2026 en representación de Grupo Andes S.A.S"

**1. Resultado con el modelo NER general (`MMG/xlm-roberta-large-ner-spanish`):**
```
PER: 'Carlos Ramírez' (0.998)
ORG: 'Ju' (0.935)
ORG: 'z' (0.869)
ORG: 'gado Primero de' (0.873)
LOC: 'Lima' (0.473)
ORG: 'Grupo Andes S' (0.997)
ORG: 'A.S' (0.990)
```

**2. Resultado con el modelo NER legal especializado (`agomez302/nlp-dr-ner`):**
```
DATE: '12 de mayo de 2026' (0.999)
```

**3. Comparación de resultados:**

| Entidad | Modelo general | Modelo legal especializado |
|---|---|---|
| Carlos Ramírez (abogado) | ✅ PER (0.998) | ❌ No detectado |
| Juzgado Primero de Lima (institución) | ⚠️ Fragmentado en 3 `ORG` ("Ju", "z", "gado Primero de") + `LOC` "Lima" con score bajo (0.47) | ❌ No detectado |
| Grupo Andes S.A.S (organización) | ⚠️ Fragmentado en 2 entidades ("Grupo Andes S" y "A.S") | ❌ No detectado |
| 12 de mayo de 2026 (fecha) | ❌ No detectado | ✅ DATE (0.999) |

**4. Análisis de fallas de segmentación (modelo general):**
El modelo general repite aquí el mismo problema de tokenización visto en el Ejercicio 3 con "Clínica San Rafael": nombres propios compuestos por palabras poco frecuentes en el corpus CoNLL-2002 (noticias) se fragmentan en sub-tokens sueltos:
- "Juzgado Primero de Lima" se rompe en **cuatro entidades separadas** ("Ju" + "z" + "gado Primero de" + "Lima"), todas con scores más bajos que el resto del texto (el de "Lima" cae hasta 0.47, evidenciando que el modelo "duda" al reensamblar la palabra "Juzgado").
- "Grupo Andes S.A.S" se parte en dos ("Grupo Andes S" / "A.S"), separando el sufijo societario del nombre de la empresa.

Esto confirma que el modelo general **no tiene un concepto de "institución judicial"** — simplemente intenta encajar los fragmentos en las etiquetas que conoce (ORG/LOC), sin coherencia semántica sobre lo que es un juzgado.

**5. Análisis del modelo legal especializado:**
El modelo `agomez302/nlp-dr-ner` solo devolvió **una entidad: la fecha** ("12 de mayo de 2026") con un score altísimo (0.999) y una etiqueta `DATE` que el modelo general ni siquiera contempla en su esquema. Sin embargo, **no reconoció ni a la persona, ni al juzgado, ni a la organización** — es decir, su esquema de etiquetas parece estar enfocado casi exclusivamente en fechas (quizás por haber sido entrenado con documentos de otro tipo de sistema legal, dado que "dr" en el nombre del modelo sugiere **República Dominicana**, una jurisdicción distinta a la peruana del texto de prueba).

**6. Entidades importantes para revisión legal, y su cobertura real:**

| Entidad relevante | ¿Detectada? | Por cuál modelo |
|---|---|---|
| Persona (abogado) | ✅ | Solo el general |
| Institución judicial (Juzgado) | ⚠️ Parcial y fragmentada | Solo el general, mal etiquetada como ORG/LOC |
| Organización (parte representada) | ⚠️ Parcial y fragmentada | Solo el general |
| Fecha del proceso | ✅ | Solo el especializado |
| Tipo de proceso ("demanda") | ❌ | Ninguno |
| Número de expediente | ❌ (no está en el texto) | — |
| Referencia normativa | ❌ (no está en el texto) | — |

**7. Conclusión para cumplimiento:**
Este experimento demuestra, con evidencia empírica, que **ningún modelo por sí solo cubre todas las entidades relevantes de un documento legal**:
- El modelo general aporta personas y (con fallas de segmentación) organizaciones/instituciones, pero **no fechas** y **rompe nombres compuestos** como "Juzgado Primero de Lima", generando anonimizaciones inconsistentes si se usara tal cual.
- El modelo legal especializado aporta la fecha con altísima confianza, pero **ignora completamente** personas y organizaciones — posiblemente porque su dominio de entrenamiento (jurisdicción distinta, quizás República Dominicana) no coincide con el tipo de texto de prueba (proceso judicial peruano).
- La fragmentación de "Juzgado Primero de Lima" y "Grupo Andes S.A.S" es un riesgo real de cumplimiento: si se anonimiza automáticamente reemplazando cada `entity_group` detectado, el texto resultante quedaría con etiquetas repetidas y mal formadas (ej. "[ORG][ORG][ORG] de [LOC]"), lo cual no es utilizable en un pipeline de producción sin post-procesamiento adicional.
- Esto refuerza la necesidad de: (a) un modelo NER legal entrenado específicamente en el corpus jurídico de la jurisdicción objetivo (Perú, en este caso), (b) reglas de re-ensamblado de tokens fragmentados antes de aplicar cualquier lógica de anonimización, y (c) reglas complementarias (regex/gazetteers) para instituciones judiciales conocidas y para identificadores como número de expediente, ausentes en ambos modelos.

<a id="sec-ejercicio-7"></a>
## 10. Ejercicio 7 – Selección autónoma de modelo y mini-experimento

En este punto, cada grupo debe:

1. Elegir un modelo NER diferente en español desde el catálogo:  
   - [https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner](https://huggingface.co/models?pipeline_tag=token-classification&language=es&search=ner)
2. Justificar su elección según:
   - Idioma y dominio (general, biomédico, legal, etc.).  
   - Tamaño del modelo y viabilidad práctica.  
   - Documentación disponible.
   - fecha de publicación del modelo o actulización registrada

### 10.1. Plantilla de código

sugerencia de código:

In [23]:
from transformers import pipeline

# Reemplaza esta línea con el modelo que hayan elijido
modelo_estudiante = "MMG/xlm-roberta-large-ner-spanish"

ner_est = pipeline(
    "ner",
    model=modelo_estudiante,
    aggregation_strategy="simple"
)

texto_prueba = (
    "Juan Pérez autorizó el tratamiento en la Clínica del Norte de Medellín "
    "y firmó el consentimiento informado."
)

ner_est(texto_prueba)

Device set to use cpu


[{'entity_group': 'PER',
  'score': np.float32(0.99884516),
  'word': 'Juan Pérez',
  'start': 0,
  'end': 10},
 {'entity_group': 'LOC',
  'score': np.float32(0.95975435),
  'word': 'C',
  'start': 41,
  'end': 42},
 {'entity_group': 'LOC',
  'score': np.float32(0.977285),
  'word': 'lí',
  'start': 42,
  'end': 44},
 {'entity_group': 'LOC',
  'score': np.float32(0.8209004),
  'word': 'nica del Norte de',
  'start': 44,
  'end': 61},
 {'entity_group': 'LOC',
  'score': np.float32(0.7416687),
  'word': 'Medellín',
  'start': 62,
  'end': 70}]

### Modelo elegido: `mrm8488/bert-spanish-cased-finetuned-ner`

**Justificación:**

- **Idioma:** español nativo, basado en BETO (BERT entrenado desde cero en español).
- **Dominio:** general (nombres, lugares, organizaciones), entrenado sobre corpus CoNLL-2002 en español, lo que permite comparar directamente frente a MMG (también CoNLL-2002) y frente a los modelos biomédico/legal ya probados en el notebook.
- **Tamaño:** BERT-base (~110M parámetros), notablemente más liviano que XLM-RoBERTa-large, viable en CPU.
- **Documentación:** modelo público y ampliamente utilizado en la comunidad de Hugging Face para tareas de NER en español, con tarjeta de modelo y ejemplos de uso disponibles.
- **Fecha:** modelo consolidado y estable, con historial de descargas y uso continuo en proyectos NER en español.

In [29]:
from transformers import pipeline

modelo_estudiante = "mrm8488/bert-spanish-cased-finetuned-ner"

ner_est = pipeline(
    "ner",
    model=modelo_estudiante,
    aggregation_strategy="simple"
)

texto_prueba = (
    "Juan Pérez autorizó el tratamiento en la Clínica del Norte de Medellín "
    "y firmó el consentimiento informado."
)

resultado_estudiante = ner_est(texto_prueba)
print("=== Texto de prueba ===")
print(texto_prueba)
print("\n=== Resultado NER (mrm8488/bert-spanish-cased-finetuned-ner) ===")
print(resultado_estudiante)

Some weights of the model checkpoint at mrm8488/bert-spanish-cased-finetuned-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


=== Texto de prueba ===
Juan Pérez autorizó el tratamiento en la Clínica del Norte de Medellín y firmó el consentimiento informado.

=== Resultado NER (mrm8488/bert-spanish-cased-finetuned-ner) ===
[{'entity_group': 'PER', 'score': np.float32(0.9993651), 'word': 'Juan Pérez', 'start': 0, 'end': 10}, {'entity_group': 'LOC', 'score': np.float32(0.8800003), 'word': 'Clínica del Norte', 'start': 41, 'end': 58}, {'entity_group': 'LOC', 'score': np.float32(0.86696154), 'word': 'Medellín', 'start': 62, 'end': 70}]


### 10.2. Preguntas de análisis

**Texto de prueba:** "Juan Pérez autorizó el tratamiento en la Clínica del Norte de Medellín y firmó el consentimiento informado."

**Resultado obtenido (`mrm8488/bert-spanish-cased-finetuned-ner`):**
```
PER: 'Juan Pérez' (0.999)
LOC: 'Clínica del Norte' (0.880)
LOC: 'Medellín' (0.867)
```

- **¿Qué etiquetas (`entity_group`) devuelve el modelo?**  
El modelo devuelve dos tipos de etiquetas: **PER** (persona), para "Juan Pérez" con un score muy alto (0.999), y **LOC** (lugar), aplicada tanto a "Clínica del Norte" (0.880) como a "Medellín" (0.867). No aparecen etiquetas `ORG` ni `MISC` en este texto, aunque el modelo sí las contempla en su esquema (está entrenado sobre CoNLL-2002, igual que MMG).

- **¿Coinciden con las que esperaban por el dominio (general, clínico, legal, etc.)?**  
Parcialmente. Al ser un modelo **general** (dominio noticioso, CoNLL-2002), era esperable que detectara correctamente a la persona (PER) y reconociera algún tipo de lugar/institución. Sin embargo, hay un problema de **clasificación semántica**: "Clínica del Norte" es una institución de salud y debería idealmente etiquetarse como `ORG` (organización/institución), no como `LOC` (lugar geográfico). Esto confirma la limitación ya vista con otros modelos generales (como en el Ejercicio 3 con "Clínica San Rafael"): el modelo no distingue bien entre una institución y una ubicación física cuando el nombre propio contiene palabras como "Clínica". Además, y como es esperable en un modelo general, **no detecta el concepto "tratamiento"** ni **"consentimiento informado"** como entidades relevantes de dominio clínico/legal — estos son justamente el tipo de dato sensible que un modelo biomédico o legal especializado sí podría capturar.

- **¿Cómo cambiarían la estrategia de anonimización dependiendo del tipo de entidades detectadas?**  
  - Si solo se contara con este modelo general, la anonimización reemplazaría "Juan Pérez" → `[PER]`, "Clínica del Norte" → `[LOC]` y "Medellín" → `[LOC]`. Esto anonimizaría correctamente al paciente y la ciudad, pero **etiquetaría mal la institución de salud** como si fuera un lugar geográfico genérico, lo cual podría ser confuso en un reporte de auditoría (no queda claro que se trató de una institución médica).
  - Para una estrategia de anonimización robusta en un caso real de salud, se debería:
    1. Complementar con un modelo NER **biomédico** para detectar "tratamiento" y "consentimiento informado" como conceptos clínicos/procesuales (similar a lo hecho en el Ejercicio 5 con `PROC`/`CHEM`).
    2. Aplicar una regla de **post-procesamiento** que reclasifique entidades `LOC` que contengan palabras clave como "Clínica", "Hospital", "Centro Médico" hacia una etiqueta más apropiada (`ORG` o una etiqueta custom `HEALTH_FACILITY`), para no confundir instituciones con ubicaciones puras.
    3. Añadir reglas complementarias (regex) para detectar frases como "consentimiento informado", que son relevantes para cumplimiento (evidencian que se trató un dato de salud con autorización), pero que ningún modelo NER genérico reconoce como entidad.
  - En resumen: el tipo de entidad detectada (PER vs. LOC vs. ausencia total de detección) determina si basta con un reemplazo simple por etiqueta, o si se necesita reclasificación semántica y modelos adicionales antes de considerar el texto verdaderamente anonimizado para fines de cumplimiento.

<a id="sec-ejercicio-8"></a>
## 11. Ejercicio 8 – Caso de estudio de cumplimiento

**Objetivo:**  
Diseñar un mini caso realista que conecte NER con cumplimiento normativo en un dominio elegido por cada grupo.

### 11.1. Dominios sugeridos

- **Salud**: historias clínicas, recetas, informes de laboratorio.
- **Legal**: contratos, demandas, resoluciones, expedientes.
- **Educación**: listados de estudiantes, notas, informes de desempeño.
- **Finanzas**: formularios, extractos, reclamaciones, reportes de riesgo.

### 11.2. Entregable sugerido

Para cada grupo, el caso debe incluir:

1. **Texto de ejemplo** del dominio (inventado o anonimizado para clase).  
2. **Modelo NER utilizado** y breve justificación.  
3. **Salida NER** (lista de entidades).  
4. **Texto anonimizado** usando una función tipo `anonimizar_entidades`.  
5. **Riesgos identificados**: qué datos personales o sensibles aparecen, cuáles podrían faltar.  
6. **Limitaciones del modelo**: tipos de entidades que no detecta o que clasifica mal.  
7. **Relación con GDPR o con la norma local** (Colombia, México, Perú):  
   - ¿Qué principios de tratamiento se ven implicados (minimización, seguridad, finalidad, etc.)?

El propósito de este ejercicio es sirvir como puente entre la parte técnica (modelos NER) y la práctica profesional (arquitecturas de cumplimiento con LLM).

<a id="sec-recursos"></a>
## 12. Recursos y datasets recomendados

### 12.1. NER general en español

- Modelo `MMG/xlm-roberta-large-ner-spanish` (CoNLL-2002 ES):  
  - Página del modelo: https://huggingface.co/MMG/xlm-roberta-large-ner-spanish
- Introducción a token classification / NER en la Hugging Face course:  
  - https://huggingface.co/learn/llm-course/chapter7/2

### 12.2. Salud / medicina

- Repositorio de modelos biomédicos en español (PlanTL-GOB-ES):  
  - https://github.com/PlanTL-GOB-ES/lm-biomedical-clinical-es
- Datasets en Hugging Face:  
  - CANTEMIST NER: https://huggingface.co/datasets/PlanTL-GOB-ES/cantemist-ner
  - PharmaCoNER: https://huggingface.co/datasets/PlanTL-GOB-ES/pharmaconer

### 12.3. Legal

- Modelo NER legal en español:  
  - `agomez302/nlp-dr-ner`: https://huggingface.co/agomez302/nlp-dr-ner
- Guía de fine-tuning para clasificación de tokens en datos legales:  
  - https://huggingface.co/blog/bikashpatra/legal-data-token-classification-fine-tuning

### 12.4. Marco regulatorio

- GDPR – información general y guías:
  - https://gdpr.eu/what-is-gdpr/  
  - Ejemplos de contenidos para formación en GDPR: https://usercentrics.com/knowledge-hub/gdpr-training/
- Colombia – Ley 1581 de 2012:
  - Texto oficial: https://www.alcaldiabogota.gov.co/sisjur/normas/Norma1.jsp?i=49981  
- México – LFPDPPP: 
  - Guía de cumplimiento: https://resguard-solutions.com/blog/en/mexico-lfpdppp-data-protection-guide/  
- Perú – Ley 29733:
  - Guía de cumplimiento: https://resguard-solutions.com/blog/en/peru-law-29733-data-protection-guide/

<a id="sec-cierre"></a>
## 13. Cierre y reflexión final

En este notebook hemos visto cómo:

- Un modelo NER en español puede detectar entidades como personas, organizaciones y lugares en textos reales.
- Esa detección se puede usar para **anonimizar** textos antes de enviarlos a un LLM o a otra API, apoyando principios de **minimización** y **seguridad** recogidos en GDPR y en las leyes locales de Colombia, México y Perú.
- Los modelos generales tienen límites importantes, especialmente en dominios sensibles como salud y legal, donde suelen ser necesarios modelos NER **especializados** entrenados con datasets clínicos o jurídicos.
- Además de NER, usamos la tarea de **rellenar máscaras** (`fill-mask`) para observar cómo un modelo de lenguaje completa frases con huecos y cómo esto puede revelar **sesgos estadísticos** o hacer que ciertos datos enmascarados sean demasiado fáciles de adivinar.

La lección central es que la técnica por sí sola no garantiza cumplimiento:

- NER y `fill-mask` son **herramientas** dentro de un sistema más amplio que debe incluir políticas internas, formación, roles claros, contratos adecuados con proveedores y auditorías periódicas.
- Desde la perspectiva de diseño de aplicaciones de IA con LLM, el desafío está en combinar modelos, reglas de negocio y conocimiento legal para construir sistemas que sean útiles, pero también **seguros, no discriminatorios y alineados con la normativa vigente**.